# 心思维｜已有 TorchCREPE 与 VocalParse 音高交叉验证

本 Notebook 只使用 Google Drive 中已经保存的 `vocal_pitch_raw.npz`，以及仓库中已保存的 25 秒 VocalParse 测试样本。**不会运行 VocalParse、CREPE，不下载模型，也不申请 GPU。**

已改为按固定提交下载所需的 4 个小型项目文件，不再执行 `git clone`，避免 Colab 克隆分支失败。

In [ ]:
from google.colab import drive
from pathlib import Path
import importlib.util
import json
import urllib.request

# Drive 已挂载时会提示 already mounted，这是正常信息
drive.mount('/content/drive')

# 固定到已核对过的项目提交，只下载必要源文件，不克隆整个仓库
COMMIT = '1d5ad228e5035ce63820dc57d5a8114e0a901f99'
BASE_URL = f'https://raw.githubusercontent.com/Trifida/xinsw-music-transcriber/{COMMIT}'
repo_dir = Path('/content/xinsw-vocalparse-crosscheck')
files = {
    'convert_pitch_npz.py': 'scripts/convert_pitch_npz.py',
    'vocalparse.py': 'musicmaster/core/vocalparse.py',
    'vocalparse_diagnostics.py': 'musicmaster/core/vocalparse_diagnostics.py',
    'vocalparse_25s_sample.json': 'tests/fixtures/vocalparse_25s_sample.json',
}
repo_dir.mkdir(parents=True, exist_ok=True)
for local_name, repo_path in files.items():
    target = repo_dir / local_name
    request = urllib.request.Request(
        f'{BASE_URL}/{repo_path}',
        headers={'User-Agent': 'XinsW-Colab-CrossCheck'}
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            target.write_bytes(response.read())
    except Exception as exc:
        raise RuntimeError(f'下载项目文件失败：{repo_path}；原因：{exc}') from exc
print('必要项目文件下载完成：', len(files), '个')

def load_module(name, file_path):
    spec = importlib.util.spec_from_file_location(name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

converter = load_module('xinsw_npz_converter', repo_dir / 'convert_pitch_npz.py')
vocalparse = load_module('xinsw_vocalparse_adapter', repo_dir / 'vocalparse.py')
diagnostics = load_module('xinsw_pitch_diagnostics', repo_dir / 'vocalparse_diagnostics.py')

drive_root = Path('/content/drive/MyDrive/XinsW_Lead_Transcription/TorchCREPE_Test')
npz_path = drive_root / 'vocal_pitch_raw.npz'
track_json = drive_root / 'reference_pitch_track.json'
comparison_json = drive_root / 'vocalparse_pitch_comparison_30_55s.json'
vocalparse_json = repo_dir / 'vocalparse_25s_sample.json'

if not npz_path.is_file():
    raise FileNotFoundError(f'Drive 中找不到已有音高文件：{npz_path}')

# 已确认：pitch 是 Hz，periodicity 是 0-1 指标，hop=160/16000=0.01 秒
converted = converter.convert_npz_to_pitch_json(
    npz_path, track_json,
    hop_seconds=0.01,
    frequency_key='pitch',
    confidence_key='periodicity',
)
print('参考音高轨迹转换完成')
print('帧数：', converted['frame_count'])
print('时间步长：', converted['timing']['hop_seconds'], '秒')
print('文件：', track_json)

parsed = json.loads(vocalparse_json.read_text(encoding='utf-8'))
events = vocalparse.vocalparse_tokens_to_events(parsed)
report = diagnostics.compare_vocalparse_to_pitch_track(
    events,
    parsed['bpm'],
    converted['frames'],
    confidence_threshold=0.5,
    central_fraction=0.6,
    reference_time_offset_sec=30.0,
)
comparison_json.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('\n===== 交叉验证汇总 =====')
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('\n===== 有音高偏差的音符（按绝对半音误差排序）=====')
rows = []
for event, row in zip(events, report['events']):
    if not event['is_rest'] and row['pitch_error_semitones'] is not None:
        rows.append({
            '字': event.get('lyric') or event.get('raw_word'),
            'VocalParse MIDI': row['vocalparse_pitch_midi'],
            'TorchCREPE MIDI中位数': round(row['reference_median_pitch_midi'], 2),
            '误差(半音)': round(row['pitch_error_semitones'], 2),
            '疑似八度错误': row['is_octave_mismatch'],
            '有效帧数': row['voiced_frame_count'],
        })
rows.sort(key=lambda item: abs(item['误差(半音)']), reverse=True)
for row in rows[:15]:
    print(row)

print('\n完整比较报告：', comparison_json)
print('没有重新运行任何模型；periodicity 阈值 0.5 仅作筛选，不代表人工真值。')
